In [1]:
!git clone https://github.com/mkobycheva/recommendation-system.git 2>/dev/null \
  || (cd recommendation-system && git pull)

%cd recommendation-system
!git checkout transformer && git pull
!pip install -r requirements.txt -q

import sys
sys.path.insert(0, '.')

from google.colab import drive
drive.mount('/content/drive')

Already up to date.
/content/recommendation-system
Already on 'transformer'
Your branch is up to date with 'origin/transformer'.
Already up to date.
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os
import numpy as np
import pandas as pd

from src.evaluation.metrics import ndcg_at_k, recall_at_k, map_at_k

# Data preparation

In [3]:
DATA_DIR = '/content/drive/MyDrive/recsys-data'

books_train = pd.read_csv(f'{DATA_DIR}/books_train.csv')
books_valid = pd.read_csv(f'{DATA_DIR}/books_valid.csv')
books_test = pd.read_csv(f'{DATA_DIR}/books_test.csv')

movies_train = pd.read_csv(f'{DATA_DIR}/movies_train.csv')
movies_valid = pd.read_csv(f'{DATA_DIR}/movies_valid.csv')
movies_test = pd.read_csv(f'{DATA_DIR}/movies_test.csv')

In [4]:
from src.data.build_matrix import add_domain_item_ids
from sklearn.preprocessing import LabelEncoder

books_train = add_domain_item_ids(books_train, 'books')
books_valid = add_domain_item_ids(books_valid, 'books')
books_test  = add_domain_item_ids(books_test, 'books')

movies_train = add_domain_item_ids(movies_train, 'movies')
movies_valid = add_domain_item_ids(movies_valid, 'movies')
movies_test  = add_domain_item_ids(movies_test, 'movies')

train_df = pd.concat([books_train, movies_train], ignore_index=True)
valid_df = pd.concat([books_valid, movies_valid], ignore_index=True)
test_df  = pd.concat([books_test, movies_test], ignore_index=True)

encoder = LabelEncoder()
encoder.fit(pd.concat([train_df['item_id'], valid_df['item_id'], test_df['item_id']]).unique())

for df in (train_df, valid_df, test_df):
    df['item_idx'] = encoder.transform(df['item_id']) + 1  # 0 = padding

NUM_ITEMS = len(encoder.classes_) + 1
print(f"NUM_ITEMS = {NUM_ITEMS}")

NUM_ITEMS = 608159


In [5]:
MAX_LEN = 50

def build_user_sequences(df):
    """user_id -> list of item_idx, sorted by timestamp"""
    df_sorted = df.sort_values(['user_id', 'timestamp'])
    return df_sorted.groupby('user_id')['item_idx'].apply(list).to_dict()

def generate_sliding_windows(user_sequences, max_len=MAX_LEN, min_len=2):
    """
    Next-item prediction на кожній позиції одразу (SASRec-style).
    input:  [0,...,0, i1,i2,i3,i4]
    target: [0,...,0, i2,i3,i4,i5]
    """
    X, Y = [], []
    for seq in user_sequences.values():
        if len(seq) < min_len:
            continue
        inp = seq[:-1]
        tgt = seq[1:]
        if len(inp) >= max_len:
            inp = inp[-max_len:]
            tgt = tgt[-max_len:]
        else:
            pad = max_len - len(inp)
            inp = [0] * pad + inp
            tgt = [0] * pad + tgt
        X.append(inp)
        Y.append(tgt)
    return np.array(X), np.array(Y)

train_sequences = build_user_sequences(train_df)
X_train, Y_train = generate_sliding_windows(train_sequences)

valid_combined = pd.concat([train_df, valid_df])
valid_sequences = build_user_sequences(valid_combined)
X_val, Y_val = generate_sliding_windows(valid_sequences)

print(f"Train: X={X_train.shape}, Y={Y_train.shape}")
print(f"Valid: X={X_val.shape}, Y={Y_val.shape}")

Train: X=(127188, 50), Y=(127188, 50)
Valid: X=(127188, 50), Y=(127188, 50)


In [6]:
non_pad_train = (Y_train != 0).sum()
total_train = Y_train.size
print(f"Non-padding targets: {non_pad_train} out of {total_train} ({100*non_pad_train/total_train:.1f}%)")

Non-padding targets: 2696531 out of 6359400 (42.4%)


# SASRec

In [7]:
import torch
from torch.utils.data import Dataset, DataLoader

class SASRecDataset(Dataset):
    """Wraps padded (input, target) sequences for next-item prediction."""

    def __init__(self, X, Y):
        self.X = torch.tensor(X, dtype=torch.long)
        self.Y = torch.tensor(Y, dtype=torch.long)

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]


BATCH_SIZE = 128
train_loader = DataLoader(SASRecDataset(X_train, Y_train), batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(SASRecDataset(X_val, Y_val), batch_size=BATCH_SIZE, shuffle=False)

for x_batch, y_batch in train_loader:
    print(f"X batch shape: {x_batch.shape}, Y batch shape: {y_batch.shape}")
    break

X batch shape: torch.Size([128, 50]), Y batch shape: torch.Size([128, 50])


In [8]:
import torch.nn as nn

class SASRec(nn.Module):
    def __init__(self, num_items, max_len=MAX_LEN, d_model=64, n_heads=2, n_layers=2, dropout=0.2):
        super().__init__()
        self.item_emb = nn.Embedding(num_items, d_model, padding_idx=0)
        self.pos_emb = nn.Embedding(max_len, d_model)
        self.emb_dropout = nn.Dropout(dropout)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True, activation='gelu',
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.layer_norm = nn.LayerNorm(d_model)
        self.max_len = max_len

    def forward(self, item_seq):
        batch_size, seq_len = item_seq.shape
        positions = torch.arange(seq_len, device=item_seq.device).unsqueeze(0).expand(batch_size, -1)

        x = self.item_emb(item_seq) + self.pos_emb(positions)
        x = self.emb_dropout(x)

        causal_mask = nn.Transformer.generate_square_subsequent_mask(seq_len).to(item_seq.device)
        padding_mask = (item_seq == 0)

        x = self.encoder(x, mask=causal_mask, src_key_padding_mask=padding_mask)

        # padding-only query positions can produce NaN internally (fully-masked attention row);
        # those positions are excluded from the loss anyway, so zeroing NaNs here is safe
        # and stops them from contaminating real positions in later computations
        x = torch.nan_to_num(x, nan=0.0)

        x = self.layer_norm(x)
        return x

    def predict_logits(self, hidden_states):
        return hidden_states @ self.item_emb.weight.T

In [9]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = SASRec(num_items=NUM_ITEMS, max_len=MAX_LEN, d_model=32, n_heads=2, n_layers=2, dropout=0.4).to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='min', patience=1, factor=0.5)

print(f"Device: {device}")
print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

Device: cuda
Model parameters: 19,488,160


In [10]:
def sampled_logits_and_labels(model, hidden_valid, targets_valid, num_items, num_negatives=100):
    """
    hidden_valid:   (N, d_model) - model output only for real (non-padding) positions
    targets_valid:  (N,)         - the correct next item for each position
    Returns:
      logits: (N, 1 + num_negatives) - scores for [correct_item, K random negatives]
      labels: (N,) - always zero, since the correct item is always at position 0
    """
    N = hidden_valid.size(0)
    device = hidden_valid.device

    # sample random negatives per position; exclude 0 since it's padding_idx
    neg_items = torch.randint(1, num_items, (N, num_negatives), device=device)

    # (optional but recommended) remove accidental collisions where a negative == the true target
    collision = neg_items == targets_valid.unsqueeze(1)
    if collision.any():
        neg_items[collision] = torch.randint(1, num_items, (int(collision.sum().item()),), device=device)

    candidates = torch.cat([targets_valid.unsqueeze(1), neg_items], dim=1)   # (N, 1+K)
    cand_emb = model.item_emb(candidates)                                    # (N, 1+K, d_model)

    logits = torch.bmm(cand_emb, hidden_valid.unsqueeze(-1)).squeeze(-1)     # (N, 1+K)
    labels = torch.zeros(N, dtype=torch.long, device=device)                 # correct item is always at index 0

    return logits, labels

In [11]:
@torch.no_grad()
def evaluate_map_at_k(model, hidden_valid, targets_valid, num_items, num_negatives=100, k=5):
    """
    Builds candidate-pool rankings and scores them with the repo's map_at_k
    (relies on average_precision_at_k under the hood).
    Each "user" here is really one (position, target) example — with a single
    relevant item — matching the interface map_at_k expects.
    """
    model.eval()
    N = hidden_valid.size(0)
    device = hidden_valid.device

    neg_items = torch.randint(1, num_items, (N, num_negatives), device=device)
    collision = neg_items == targets_valid.unsqueeze(1)
    if collision.any():
        neg_items[collision] = torch.randint(1, num_items, (int(collision.sum().item()),), device=device)

    candidates = torch.cat([targets_valid.unsqueeze(1), neg_items], dim=1)   # (N, 1+K)
    cand_emb = model.item_emb(candidates)
    logits = torch.bmm(cand_emb, hidden_valid.unsqueeze(-1)).squeeze(-1)      # (N, 1+K)

    top_k_idx = logits.topk(k, dim=1).indices                                # (N, k), indices into candidate axis
    top_k_items = torch.gather(candidates, 1, top_k_idx)                     # (N, k), actual item ids

    recommended_items_by_user = {i: top_k_items[i].tolist() for i in range(N)}
    relevant_items_by_user = {i: [targets_valid[i].item()] for i in range(N)}

    return map_at_k(recommended_items_by_user, relevant_items_by_user, k=k)

In [12]:
@torch.no_grad()
def sampled_ranking_map_at_k(model, hidden_all, targets_all, num_items, num_negatives=999, k=5, seed=42):
    """
    Middle ground between the 100-negative sampled pool (too easy) and
    full-catalog ranking (too expensive/too hard early in training).
    Fixed seed makes the candidate pool identical every epoch, so MAP@5
    is comparable across epochs (not noisy due to different random negatives).
    """
    model.eval()
    device = hidden_all.device
    N = hidden_all.size(0)

    generator = torch.Generator(device=device).manual_seed(seed)
    neg_items = torch.randint(1, num_items, (N, num_negatives), device=device, generator=generator)

    collision = neg_items == targets_all.unsqueeze(1)
    if collision.any():
        neg_items[collision] = torch.randint(1, num_items, (int(collision.sum().item()),), device=device, generator=generator)

    candidates = torch.cat([targets_all.unsqueeze(1), neg_items], dim=1)   # (N, 1+K)
    cand_emb = model.item_emb(candidates)
    logits = torch.bmm(cand_emb, hidden_all.unsqueeze(-1)).squeeze(-1)      # (N, 1+K)

    top_k_idx = logits.topk(k, dim=1).indices
    top_k_items = torch.gather(candidates, 1, top_k_idx)

    recommended_items_by_user = {i: top_k_items[i].tolist() for i in range(N)}
    relevant_items_by_user = {i: [targets_all[i].item()] for i in range(N)}

    return map_at_k(recommended_items_by_user, relevant_items_by_user, k=k)

## Training

In [13]:
import wandb

wandb.init(
    project="recsys-sequential",
    name="sasrec-v2",
    config={
        "model": "SASRec",
        "max_len": MAX_LEN,
        "d_model": 64,
        "n_heads": 2,
        "n_layers": 2,
        "dropout": 0.2,
        "batch_size": BATCH_SIZE,
        "lr": 1e-3,
        "weight_decay": 1e-5,
    }
)

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: dkalchenko (dkalchenko-kyiv-school-of-economics) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [14]:
EPOCHS = 30
PATIENCE = 3
best_val_map = -float('inf')
patience_counter = 0
best_state = None

for epoch in range(EPOCHS):
    model.train()
    train_loss = 0.0
    total_grad_norm = 0.0

    for x_batch, y_batch in train_loader:
        x_batch, y_batch = x_batch.to(device), y_batch.to(device)
        optimizer.zero_grad()
        hidden = model(x_batch)

        # Flatten and keep only non-padding positions BEFORE the expensive matmul
        hidden_flat = hidden.view(-1, hidden.size(-1))
        targets_flat = y_batch.view(-1)
        mask = targets_flat != 0
        hidden_valid = hidden_flat[mask]
        targets_valid = targets_flat[mask]

        logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
        loss = criterion(logits, labels)
        loss.backward()

        grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=2.0)
        total_grad_norm += grad_norm.item()

        optimizer.step()
        train_loss += loss.item()

    train_loss /= len(train_loader)
    avg_grad_norm = total_grad_norm / len(train_loader)

    model.eval()
    val_loss = 0.0
    all_hidden = []
    all_targets = []

    with torch.no_grad():
        for x_batch, y_batch in val_loader:
            x_batch, y_batch = x_batch.to(device), y_batch.to(device)
            hidden = model(x_batch)

            hidden_flat = hidden.view(-1, hidden.size(-1))
            targets_flat = y_batch.view(-1)
            mask = targets_flat != 0
            hidden_valid = hidden_flat[mask]
            targets_valid = targets_flat[mask]

            all_hidden.append(hidden_valid.cpu())
            all_targets.append(targets_valid.cpu())

            logits, labels = sampled_logits_and_labels(model, hidden_valid, targets_valid, NUM_ITEMS, num_negatives=100)
            loss = criterion(logits, labels)
            val_loss += loss.item()

    val_loss /= len(val_loader)

    all_hidden = torch.cat(all_hidden).to(device)
    all_targets = torch.cat(all_targets).to(device)
    val_map5 = sampled_ranking_map_at_k(model, all_hidden, all_targets, NUM_ITEMS, num_negatives=999, k=5, seed=42)

    scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    wandb.log({
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_loss": val_loss,
        "grad_norm": avg_grad_norm,
        "learning_rate": current_lr,
        "val_map@5": val_map5,
    })

    print(f"Epoch [{epoch+1}/{EPOCHS}] Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val MAP@5: {val_map5:.4f} | Grad Norm: {avg_grad_norm:.4f}")

    if val_map5 > best_val_map:
        best_val_map = val_map5
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        patience_counter = 0
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f"Early stopping at epoch {epoch+1}")
            break

model.load_state_dict(best_state)
wandb.finish()

/usr/local/lib/python3.12/dist-packages/torch/nn/modules/transformer.py:431: UserWarning: Support for mismatched src_key_padding_mask and mask is deprecated. Use same type for both instead.
  src_key_padding_mask = F._canonical_mask(


OutOfMemoryError: CUDA out of memory. Tried to allocate 21.74 GiB. GPU 0 has a total capacity of 14.56 GiB of which 13.69 GiB is free. Including non-PyTorch memory, this process has 888.00 MiB memory in use. Of the allocated memory 695.03 MiB is allocated by PyTorch, and 52.97 MiB is reserved by PyTorch but unallocated. If reserved but unallocated memory is large try setting PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True to avoid fragmentation.  See documentation for Memory Management  (https://docs.pytorch.org/docs/stable/notes/cuda.html#optimizing-memory-usage-with-pytorch-cuda-alloc-conf)